In [112]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/tabular-playground-series-apr-2022/sample_submission.csv
/kaggle/input/competitions/tabular-playground-series-apr-2022/train_labels.csv
/kaggle/input/competitions/tabular-playground-series-apr-2022/train.csv
/kaggle/input/competitions/tabular-playground-series-apr-2022/test.csv


In [113]:
train = pd.read_csv('/kaggle/input/competitions/tabular-playground-series-apr-2022/train.csv')
test = pd.read_csv('/kaggle/input/competitions/tabular-playground-series-apr-2022/test.csv')
labels = pd.read_csv('/kaggle/input/competitions/tabular-playground-series-apr-2022/train_labels.csv')
sub = pd.read_csv('/kaggle/input/competitions/tabular-playground-series-apr-2022/sample_submission.csv')

In [114]:
# 1. check
print(train.groupby('sequence').size().value_counts())
print(train['subject'].nunique(), test['subject'].nunique())
print(labels['state'].value_counts(normalize='True'))
print(train.isnull().sum())

60    25968
Name: count, dtype: int64
672 319
state
1    0.501155
0    0.498845
Name: proportion, dtype: float64
sequence     0
subject      0
step         0
sensor_00    0
sensor_01    0
sensor_02    0
sensor_03    0
sensor_04    0
sensor_05    0
sensor_06    0
sensor_07    0
sensor_08    0
sensor_09    0
sensor_10    0
sensor_11    0
sensor_12    0
dtype: int64


In [115]:
train.head()

,sequence,subject,step,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12
0,0,47,0,-0.196291,0.112395,1.0,0.329204,-1.004660,-0.131638,-0.127505,0.368702,-0.1,-0.963873,-0.985069,0.531893,4.751492
1,0,47,1,-0.447450,0.134454,1.0,-0.658407,0.162495,0.340314,-0.209472,-0.867176,0.2,-0.301301,0.082733,-0.231481,0.454390
2,0,47,2,0.326893,-0.694328,1.0,0.330088,0.473678,1.280479,-0.094718,0.535878,1.4,1.002168,0.449221,-0.586420,-4.736147
3,0,47,3,0.523184,0.751050,1.0,0.976991,-0.563287,-0.720269,0.793260,0.951145,-0.3,-0.995665,-0.434290,1.344650,0.429241
4,0,47,4,0.272025,1.074580,1.0,-0.136283,0.398579,0.044877,0.560109,-0.541985,-0.9,1.055636,0.812631,0.123457,-0.223359


In [116]:
labels.head()

,sequence,state
0,0,0
1,1,1
2,2,1
3,3,1
4,4,1


In [117]:
#2 feature maker
def make_features(df):
    df.sort_values(['sequence', 'step'])
    sensors = [i for i in df.columns if i.startswith('sensor')]
    features = df.groupby('sequence')[sensors].agg(['mean', 'std', 'min', 'max', 'median', 'skew'])
    features.columns = [f'{s}_features_{stat}' for s, stat in features.columns]

    diff_features_bef = df[sensors].diff().where(df['step']!=0)
    diff_features = diff_features_bef.groupby(df['sequence'])[sensors].agg(['mean', 'max', 'std'])
    #왜 groupby('sequence')는 안되는가
    diff_features.columns = [f'{s}_diff_features_{stat}' for s,stat in diff_features.columns]

    features = features.join(diff_features, rsuffix='_features')
    #features.describe()
    return features

In [118]:
#3 Dataset
X = make_features(train)
y = labels.set_index('sequence').loc[X.index, 'state'] 
### X.index가 행 state가 열인 값을 sequence를 인덱스로 설정할 때 참고(loc)하라
### 목적 : X.index (train.groupby...)의 순서를 따라가도록
groups = train.groupby('sequence')['subject'].first().loc[X.index]
### X 인덱스에 맞춰서 train을 sequence로 묶었을 때 subject(측정 대상)의 첫 번째 값으로 채운게 groups
### 60개 행 중에 하나 골라야 (X.index에 하나에 행 하나로 맞춰야 하니)-> first.. 다 똑같으므로
X_test = make_features(test).loc[sub['sequence']]
X_test.columns = X.columns

subject는 피처는 모델 입력에 넣으면 학습에 부정적 (subject와 state는 관계가 없다)이므로 제외 후 데이터를 나누는 기준으로 쓰기 위해 (KFold, 데이터 나눌 때 최소한 subject가 잘리면 안되므로) 별도로 넣는 것.

In [119]:
#본학습과 평가
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

gfk = GroupKFold(n_splits=5)
oof = np.zeros(len(X))
test_pred = np.zeros(len(X_test))

#그룹 K 폴드를 쓰는 이유는..
for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups)):
    model = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.03, verbose=-1)
    
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx], 
              eval_set=[([X.iloc[va_idx], y.iloc[va_idx]])], 
              eval_metric='auc',
              callbacks=[lgb.early_stopping(100, verbose=False)])
    
    oof[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
    test_pred = model.predict_proba(X_test)[:, 1] / gkf.n_splits
    print(f'Fold {fold} AUC: {roc_auc_score(y.iloc[va_idx], oof[va_idx]):.4f}')

print(f'CV AUC: {roc_auc_score(y, oof):.4f}')

Fold 0 AUC: 0.9137
Fold 1 AUC: 0.9135
Fold 2 AUC: 0.9273
Fold 3 AUC: 0.9236
Fold 4 AUC: 0.9246
CV AUC: 0.9204


In [120]:
sub['state'] = test_pred
sub.to_csv('submission.csv', index=False)
print(sub.head())

   sequence     state
0     25968  0.179970
1     25969  0.198454
2     25970  0.000815
3     25971  0.068908
4     25972  0.146934
